## Notebook32

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
prov = funs.read_file(ub + "data/it_province.geojson")
covid = pl.read_csv(ub + "data/it_province_covid.csv")

### Questions

This notebook uses two tables. `prov` is a GeoJSON file of Italy's 107
provinces, the administrative unit roughly equivalent to a US county. It has
one row per province with a `province` name, a `geometry` column, and the
`_crs` helper column; the geometries are polygons, not the points from last
time, so each one traces a whole boundary rather than marking a single spot.
`covid` is a daily record of new COVID-19 cases per province from February
2020 through November 2021, with columns `date`, `region`, `province`, and
`cases`. The `province` column is the shared key between the two tables.

As in the last notebook, a few answers get saved back into `prov` — the
projection step and the metric columns built on top of it — because later
questions reuse them. Anything that is not a geometry- or column-building
step should just be printed.

1. In the first block, print `prov`. In the second block, select `province` and
the first 15 characters of each `geometry` string (`c.geometry.str.slice(0, 15)`)
for the first 5 rows to see how the polygons begin. What geometry type are these,
and why can't they be drawn with `geom_point()` the way the city points were?

**Answer**:


2. Draw the provinces with `geom_map(fill="white", color="black")` and
`coord_fixed()`, in the raw coordinates the file loaded in.

The file loaded in EPSG:4326, plain longitude and latitude, which you can see
in the `_crs` column from question 1. The outline is recognizably Italy, but
the axis numbers are degrees, and with `coord_fixed()` locking one degree of
longitude to the same length as one degree of latitude, the boot comes out a
little wider and flatter than it should be — at 42°N a degree of longitude
covers noticeably less ground than a degree of latitude, and this plot
pretends they are equal.

3. Before projecting, try to compute each province's area with `prov.geo.area`
while it is still in EPSG:4326. Sort by the new `area` column, smallest first,
and print `province` and `area`. Do the numbers make sense as areas?

**Answer**:


4. In the first block, project `prov` into EPSG:7791, Italy's national reference
system, and save the result back into `prov`. In the second block, draw it again
with `geom_map(fill="white", color="black")` and `coord_fixed()`, and compare the
shape to question 2.

**Answer**:


5. Make an interactive map of the provinces with `.geo.explore(tooltip=
["province"])`. Zoom in on a province you can place from memory and check that
its boundary sits where it should.

6. Now that `prov` is projected, compute `prov.geo.area` again. In the first
block, save it back into `prov`. In the second block, print the five largest
provinces by area (`province` and `area`, sorted descending). The chapter noted
the units; what are they here?

**Answer**:


7. Print the five *smallest* provinces by area the same way. These are the
ones a national map will barely show.

**Answer**:


8. Replace each province's polygon with its centroid using the `centroid`
property (`prov.geo.centroid` — a property, so no parentheses), and print
`province` and the first 12 characters of the new `geometry`. What kind of
geometry is the result now?

**Answer**:


9. In the first block, confirm `province` is a clean key in the polygon table
with `.select(c.province.n_unique(), pl.len())`. In the second block, build a
`total` column — the sum of `cases` per province over the whole period — and
join it onto `prov`. In the third block, make a choropleth with
`geom_map(color="white")`, `coord_fixed()`, and `fill=c.total`.

**Answer**:


10. Total cases is an absolute count, and the chapter warns that choropleths of
absolute counts mislead. In the first block, build `dens`, cases per square
kilometer (`c.total / (c.area / 1_000_000)`). In the second block, make the same
choropleth with `fill=c.dens`. How does the picture change, and which map is the
more honest one?

**Answer**:


11. Confirm the density story numerically: sort `prov` by `dens`, descending,
and print the five densest provinces (`province` and `dens`). Does the top of
this list match the bright spots on the map from question 10?

**Answer**:
